# 🫀 퀘스트 46 · Q4-F — **배포 방식**, 그리고 고부담 환자의 진짜 과제

| | **MedKOS / `notebooks/quest46_q4f_deployment_mode.ipynb`** |
|---|---|
| 퀘스트 | `ailab-2026-0046` — 층② 점수 눈금 |
| 부모 런 | `quest46_q4e_operating_point`(`20260805T0413`) |
| 성격 | **처방을 고르는 런** — 지표가 아니라 **배포 방식**을 판정한다 |

## Q4-E 가 남긴 것

**모델은 무작위가 아니다.** 매크로 **AUROC 0.9418**, 그리고 유병률 사분위별 AUROC 가
0.9513 / 0.9461 / 0.9358 / **0.9340** 으로 **거의 평평**하다.

```
유병률 사분위        평균유병률   PR-AUC    lift    AUROC
[0.0070, 0.0219]     0.0150    0.4782   31.8배   0.9513
[0.0219, 0.0447]     0.0326    0.5180   15.9배   0.9461
[0.0447, 0.0980]     0.0698    0.5913    8.5배   0.9358
[0.0980, 0.5764]     0.2175    0.7214    3.3배   0.9340
```

**PR-AUC 는 오히려 오르고(0.478 → 0.721) lift 만 떨어진다.** 모델이 나빠지는 게 아니라
**무작위 기저선이 올라가는 것**이다. 레코드 48(유병률 0.5764)도 AUROC **0.8491** 이다.

## ★★★ 그런데 동작점에서 참사가 났다

```
전역 단일 문턱 q=5%      민감도   놓친 S(FN)
#14 (유병률 0.1961)      0.0000      614
#16 (0.1336)             0.0000      382
#37 (0.1687)             0.0000      460
#48 (0.5764)             0.0011    **1816**      ← AUROC 0.8491 인데 다 놓친다
```

**순위는 좋은데 절대 수준이 어긋나** 문턱을 아무도 못 넘는다. 상위 6개 레코드만으로
FN **3342** 개.

## 그리고 해법은 이미 로그 안에 있었다

```
              민감도(평균)      SD      경보율 SD
환자별 예산 5%    0.5030      0.3469     0.0000
전역 단일 문턱     0.2743      0.3438     0.0540      ← 1.83배 차이
환자별 예산 10%   0.6970      0.3167     0.0000
전역 단일 문턱     0.3873      0.3703     0.0625      ← 1.80배
```

**환자별 예산**(각 기록에서 상위 q% 를 검토 후보로 올린다)은 홀터 판독의 **표준 작업
흐름**이지 우회가 아니다. 그리고 결정적으로:

| | 환자별 예산 |
|---|---|
| raw | 0.5054 |
| A_em | 0.5054 |
| TE | 0.5030 |
| TT | 0.5030 |

**네 팔이 같다.** Q4-E 의 H2 가 증명한 대로 레코드 내 순위는 층②가 못 건드리기 때문이다.
⇒ **예산 방식으로 배포하면 사전확률 보정·부담 특징이 전부 불필요하다.**

## 고부담 환자 — 과제가 다르다

유병률 57.6% 환자에게 「이 박동이 SVEB 인가」를 묻는 건 **임상적으로 이미 답이 난 질문**
이다. 그 환자에게 필요한 건 탐지가 아니라 **부담 정량**(「SVEB 부담 X%」)이다. 그런데
이 퀘스트는 **π̂ 오차를 한 번도 지표로 재지 않았다** — Q3 은 π̂ 가 *순위에 미치는 영향*만
봤다. J4 가 그걸 처음으로 **산출물 자체로** 잰다.

## 관문 (사전등록)

| 관문 | 무엇 | 통과 기준 |
|---|---|---|
| **J0** | 코호트 · Platt 기울기 | 구성. 깨지면 **중단** |
| **J1 ★★★ 주 관문** | **같은 총 경보 예산**에서 환자별 배분 vs 전역 문턱 | 측정된 영점 상단 초과 |
| **J2 ★★★ 층② 무관성** | 예산 방식에서 `raw` ≡ `A_em` (정확히) · `TE` ≈ `raw` | 1e-12. 깨지면 **중단** |
| **J3 ★★ 임상 예산** | 분율이 아니라 **절대 개수**(기록당 100·300개) | 관문 아님 |
| **J4 ★★★ 부담 정량** | π̂ 오차 — 네 추정기의 \|π̂ − π*\| | 관문 아님. **산출물 지표** |
| **J5 ★★** | 자기 예산으로도 실패하는 환자 — 층① 진단 | 관문 아님 |
| **J6** | 결론 검산표 | R38 ⑦ · R39 ⑤ |

### 판정표

- **J1 ✅ · J2 ✅** → **배포 처방 확정**: `raw` + Platt + **환자별 예산**. 층② 전부 제거
- **J1 ❌** → 전역 문턱이 낫다 → 층②(척도 정렬)가 다시 필요하다
- **J4 에서 π̂ 오차가 크다** → 고부담 환자용 **정량 트랙**이 별도 과제로 열린다

### 사전등록 상수 (R39 ① · R34 ②)

- 분율 예산 `FLAG_Q = (0.05, 0.10)` · 절대 예산 `FLAG_K = (100, 300)` — **미리 고정**
- 전역 문턱은 **그 fold 의 DEV 에서만**. J1 은 **총 경보 수를 맞춰** 비교한다
- 주 지표 = `q = 0.05` · 보정기 `platt` · 팔 `raw`

⚠️ **새 데이터 0** — `svdb_data5.npz` 만.


In [ ]:
# CELL 0 — 공용 사전점검
import numpy as np

def decide(lo, hi, thr, direction):
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if not (np.isfinite(lo) and np.isfinite(hi) and np.isfinite(thr)):
        return "⚠️ 미결"
    if direction == ">":
        if lo > thr: return "✅ 지지"
        if hi < thr: return "❌ 기각"
    else:
        if hi < thr: return "✅ 지지"
        if lo > thr: return "❌ 기각"
    return "⚠️ 미결"

def mde(lo, hi):
    return (hi - lo) / 2.0 if np.isfinite(lo) and np.isfinite(hi) else float("nan")

def boot_mean(v, seed, nb=3000, q=2.5):
    d = np.asarray(v, float); d = d[np.isfinite(d)]
    if len(d) < 3:
        return float("nan"), float("nan"), float("nan"), len(d)
    rng = np.random.RandomState(seed)
    b = [d[rng.randint(0, len(d), len(d))].mean() for _ in range(nb)]
    return (float(d.mean()), float(np.percentile(b, q)),
            float(np.percentile(b, 100 - q)), len(d))

def boot_pair(a, b, seed, nb=3000, q=2.5):
    a = np.asarray(a, float); b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b); a, b = a[m], b[m]
    if len(a) < 3:
        return float("nan"), float("nan"), float("nan"), len(a)
    rng = np.random.RandomState(seed)
    d = [(b[j] - a[j]).mean() for j in (rng.randint(0, len(a), len(a)) for _ in range(nb))]
    return (float((b - a).mean()), float(np.percentile(d, q)),
            float(np.percentile(d, 100 - q)), len(a))

def need_super(n, half, eff, p80=False):
    if not np.isfinite(half) or not np.isfinite(eff) or abs(eff) < 1e-9 or n < 1:
        return float("nan")
    r = float(n) * (half / abs(eff)) ** 2
    return r * 2.04 if p80 else r

class AssetError(RuntimeError): pass
print("CELL 0 ✅")


In [ ]:
# CELL 1 — 설정 · 사전등록
import os, sys, json, importlib, time, warnings
importlib.invalidate_caches(); warnings.filterwarnings("ignore")

SMOKE = os.environ.get("MEDKOS_SMOKE") == "1"
_ENV_ROOT = os.environ.get("MEDKOS_DRIVE_ROOT")
if _ENV_ROOT:
    DRIVE_ROOT = _ENV_ROOT
else:
    try:
        from google.colab import drive; drive.mount("/content/drive", force_remount=False)
        DRIVE_ROOT = "/content/drive/MyDrive"
    except Exception as e:
        print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
MITBIH  = os.path.join(DRIVE_ROOT, "mitbih")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

SEED0, IDX_S = 20260805, 1
RHY_K = (5, 10, 20, 32)
MIN_S, MIN_N = 25, 25

# ── ★ 사전등록 상수 (SMOKE 가 절대 안 건드린다)
TOL_IDENT = 1e-12
DEV_EVERY = 4
FLAG_Q = (0.05, 0.10)        # 분율 예산
FLAG_K = (100, 300)          # ★ 절대 예산 — 24시간 홀터에서 5% 는 판독 불가능한 양이다
MAX_NEG_SLOPE = 0.10
MAIN_Q = 0.05

NB_BOOT = 400 if SMOKE else 2000
N_PERM  = 2   if SMOKE else 10

CAL = "platt"                # Q4-D 결론 — 순위를 보존하고 ECE 도 같거나 낫다
ARMS = ("raw", "A_em", "TE")
MAIN_ARM = "raw"             # ★ J2 가 서면 이게 처방이다(층② 없음)
PI_EST = ("em", "mean_p", "bbse", "count")
READ_ORDER = ("J0", "J1", "J2", "J3", "J4", "J5", "J6")

SV5 = os.path.join(MITBIH, "svdb_data5.npz")

REF = dict(   # Q4-E(`20260805T0413`) 실측 — 같은 코호트 · 재현 앵커
    n_ok=56, dom_rec=48, mean_prev=0.0837,
    macro_ap=0.5772, macro_auc=0.9418, xrec=0.9184,
    qband_auc=[0.9513, 0.9461, 0.9358, 0.9340],
    qband_lift=[31.8, 15.9, 8.5, 3.3],
    dom48=dict(prev=0.5764, ap=0.7489, auc=0.8491, lift=1.30),
    bud5=dict(raw=0.5054, A_em=0.5054, TE=0.5030),
    glb5=dict(raw=0.3500, A_em=0.3764, TE=0.2743),
    bud10=0.6970, glb10=0.3873,
    worst_fn=[("#14", 614), ("#16", 382), ("#37", 460), ("#48", 1816)])

RULE_CHECK = {
    "R11 매크로":       "환자 단위로 채점. 매크로는 **레코드별 기저선과 함께** 읽는다",
    "R16 fallback 없음": "자산 없으면 **중단**",
    "R22 누출 없음":     "★★ 문턱·TPR/FPR 을 **그 fold 의 DEV 에서만** 추정한다",
    "R26 / R38 ②":      "대비의 **영점**을 rep×레코드로 측정. 못 쟀으면 **안 읽는다**",
    "R29 ② 분기 금지":   "J0 · J2 가 깨지면 아래를 **안 읽는다**",
    "R33 ① MDE":        "관문마다 MDE. **미결 ≠ 등가**",
    "R34 ② 문턱 금지":  "★★★ 예산 `FLAG_Q`·`FLAG_K` 를 **사전 고정**. TEST 에서 쓸어보지 않는다",
    "R35 ① 자 먼저":    "★★ **J2 가 자다** — 예산 방식에서 층② 가 무의미함을 먼저 세운다",
    "R36 ② 선택 편의":  "★ 배포 방식을 **총 경보 수를 맞춰** 비교한다 — 많이 울려서 이기면 안 된다",
    "R40 ① λ ≠ 타당성":  "AUROC 가 좋아도 **동작점이 좋다는 보장은 없다** — Q4-E 가 그걸 봤다",
    "R41 ② 0 근처":     "효과가 0 근처면 필요표본은 해석 불가",
}

CONFIG = dict(
    exp="quest46_q4f_deployment_mode", quest="ailab-2026-0046", step="deployment-mode",
    parent_exp=["quest46_q4e_operating_point"],
    purpose=("**처방을 고르는 런.** Q4-E 가 두 가지를 동시에 보였다. (좋은 쪽) 매크로 "
             "**AUROC 0.9418** 이고 유병률 사분위별 AUROC 가 0.9513/0.9461/0.9358/0.9340 로 "
             "**거의 평평**하다 — 「고유병률에서 무의미」는 **PR-AUC 기저선이 올라가는 것**"
             "이지 모델 열화가 아니다(레코드 48 도 AUROC 0.8491). (나쁜 쪽) 그런데 **전역 "
             "단일 문턱**에서 레코드 48 이 FN **1816**, 상위 6개만으로 FN 3342 다 — 순위는 "
             "좋은데 **절대 수준이 어긋나** 문턱을 아무도 못 넘는다. ★★★ 그리고 해법이 이미 "
             "로그 안에 있었다: **환자별 예산**(각 기록에서 상위 q%)이 민감도를 0.2743 → "
             "0.5030 으로 **1.83배** 올린다. 게다가 예산 방식에서는 네 팔이 같다"
             "(raw 0.5054 = A_em 0.5054 ≈ TE 0.5030) — Q4-E 의 H2 가 증명한 대로 레코드 내 "
             "순위는 층② 가 못 건드리기 때문이다. ⇒ **예산 방식으로 배포하면 사전확률 "
             "보정·부담 특징이 전부 불필요하다.** 이 런은 그걸 **같은 총 경보 예산**에서 "
             "판정하고(J1·J2), **임상적으로 판독 가능한 절대 예산**(기록당 100·300개)으로 "
             "다시 재고(J3), 고부담 환자의 진짜 과제인 **부담 정량**(π̂ 오차)을 이 퀘스트에서 "
             "**처음으로 산출물 지표로** 잰다(J4)."),
    dataset="SVDB — svdb_data5.npz (리듬 특징만 · 새 데이터 0)",
    cal=CAL, arms=list(ARMS), main_arm=MAIN_ARM, pi_est=list(PI_EST),
    flag_q=list(FLAG_Q), flag_k=list(FLAG_K), main_q=MAIN_Q, read_order=READ_ORDER,
    dev_every=DEV_EVERY, n_boot=NB_BOOT, n_perm=N_PERM, smoke=SMOKE,
    ref=REF, rule_check=RULE_CHECK,
    predictions={
        "J0": "코호트 + Platt 기울기(중앙값 부호 · 음수 비율). 깨지면 **중단**",
        "J1": "★★★ **주 관문 — 배포 방식.** 전역 단일 문턱이 코호트 전체에서 쓴 **총 경보 "
              "수**를 그대로 환자별로 균등 배분했을 때, 레코드별 민감도가 더 높은가. "
              "**총 경보 수를 맞추므로** 「많이 울려서 이기는」 경로가 막힌다(R36 ②)",
        "J2": "★★★ **자(구성)** — 예산 방식에서 `raw` 와 `A_em` 의 레코드별 민감도가 "
              "**정확히** 같아야 한다(A_em 은 raw 의 레코드별 상수 시프트이고, 예산은 "
              "레코드 내 순위만 쓴다). 서면 **층② 를 처방에서 통째로 뺄 수 있다**. "
              "깨지면 구현 오류이므로 **중단**",
        "J3": "★★ **임상 예산** — 24시간 홀터는 박동이 10만 개라 5% 면 5000개다. **판독 "
              "불가능**하다. 기록당 **절대 100·300개**로 다시 재서 실제 검토 부담에서의 "
              "민감도·PPV 를 낸다",
        "J4": "★★★ **부담 정량(산출물 지표)** — 유병률 57.6% 환자에게 「이 박동이 SVEB 인가」는 "
              "임상적으로 이미 답이 난 질문이다. 필요한 건 **「SVEB 부담 X%」**다. 네 추정기"
              "(EM · 평균사후 · BBSE/Rogan-Gladen · 문턱초과율)의 |π̂ − π*| 를 잰다. "
              "이 퀘스트가 **한 번도 안 잰 것**이다 — Q3 은 π̂ 가 순위에 주는 영향만 봤다",
        "J5": "★★ **자기 예산으로도 실패하는 환자** — 층② 로 못 고치는 자리이므로 "
              "**층①(표현)의 과제**다. 누가·왜 실패하는지 특정한다",
        "J6": "결론 검산표"},
    caveat=("★★★ **J1 은 「총 경보 수를 맞춰」 비교한다** — 안 맞추면 많이 울리는 쪽이 "
            "민감도로 이긴다(Q4-E 스모크에서 `A_em` 이 경보율 0.2505 로 그랬다). "
            "★★ **J2 가 서면 처방이 단순해진다** — `raw` + Platt + 환자별 예산. 층② 의 "
            "사전확률 보정도, 부담 특징도, π̂ 추정도 **필요 없다**. 다만 그건 **탐지** "
            "과제에 한한 얘기이고, **정량**(J4)에는 π̂ 가 그대로 필요하다. "
            "★ **고유병률 환자의 낮은 lift 는 모델 열화가 아니다**(사분위별 AUROC 평평). "
            "다만 그 환자에게 **탐지의 임상적 가치가 낮은 것**은 사실이므로, 과제를 "
            "**정량으로 바꾸는 것**이 옳은 대응이다."))
np.random.seed(SEED0)
run = MedKOSRun("quest46_q4f_deployment_mode", CONFIG, project=PROJECT)
run.log("설정 ✅ **Q4-F — 배포 방식, 그리고 고부담 환자의 진짜 과제**")
run.log(f"  ★★★ 주 관문 = **환자별 예산 vs 전역 문턱**(총 경보 수를 맞춰서)")
run.log(f"  ★★★ J2 가 서면 처방은 **`{MAIN_ARM}` + Platt + 환자별 예산** — 층② 전부 제거")
run.log(f"  ★★ **절대 예산 {FLAG_K}** 도 잰다 — 24시간 홀터에서 5% 는 판독 불가능한 양이다")
run.log(f"  ★★★ **부담 정량(π̂ 오차)을 이 퀘스트에서 처음으로 산출물 지표로** 잰다")
run.log(f"  ▸ Q4-E 앵커 — 매크로 AUROC {REF['macro_auc']} · 사분위 AUROC {REF['qband_auc']} "
        f"(평평하다) · 레코드 48 AUROC {REF['dom48']['auc']}")
if SMOKE:
    run.log(f"  ⚠️ **스모크런** — 비용 손잡이만 축소(NB_BOOT={NB_BOOT} · N_PERM={N_PERM})")
run.log("\n  사전등록 규칙 체크리스트 (R29 ③)")
for k_, v_ in RULE_CHECK.items():
    run.log(f"    [x] {k_:<18} {v_}")


In [ ]:
# CELL 2 — 【J-0】 코호트 · LORO · 배분 방식 정의
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

run.log("\n" + "=" * 100)
run.log("【J-0】 코호트 · LORO · 배분 방식")
run.log("=" * 100)
VERD, NOTE = {}, {}
def g_(k, v, d):
    VERD[k] = v; NOTE[k] = d; run.log(f"  {k:<5}{v}  {d}")

if not os.path.exists(SV5):
    raise AssetError(f"{SV5} 없음(R16)")
D5 = np.load(SV5, allow_pickle=True)
PID = np.asarray(D5["pid"]).astype(int); Y3 = np.asarray(D5["y3"]).astype(int)
PRE = np.asarray(D5["pre_rr"], float); POST = np.asarray(D5["post_rr"], float)
K = np.where(Y3 >= 0)[0]
RID = PID[K]; TT_ = (Y3[K] == IDX_S)
pre = PRE[K].astype(float); post = POST[K].astype(float)
RS = np.array(sorted(set(RID.tolist())))

_S = pd.Series(pre); _G = _S.groupby(pd.Series(RID))
def local_base(k):
    r = np.asarray(_G.apply(lambda x: x.shift(1).rolling(k, min_periods=1).median())).astype(float)
    return np.where(np.isfinite(r), r, pre)
_med = _G.transform("median").to_numpy()
_std = _G.transform("std").to_numpy(); _mean = _G.transform("mean").to_numpy()
RHY = np.nan_to_num(np.c_[_med - pre,
                          np.column_stack([1.0 - pre / (local_base(k) + 1e-9) for k in RHY_K]),
                          post - pre, np.nan_to_num(_std / (_mean + 1e-9)),
                          np.log1p(np.clip(pre, 0, None)), np.log1p(np.clip(post, 0, None))],
                    nan=0.0, posinf=0.0, neginf=0.0)

IDXS = {int(r): np.where(RID == r)[0] for r in RS}
REC_OK = [int(r) for r in RS
          if TT_[IDXS[int(r)]].sum() >= MIN_S and (~TT_[IDXS[int(r)]]).sum() >= MIN_N]
BURD = {r: float(TT_[IDXS[r]].mean()) for r in REC_OK}
NRE = len(REC_OK); MEAN_PREV = float(np.mean([BURD[r] for r in REC_OK]))
s_all = np.array([int(TT_[IDXS[r]].sum()) for r in REC_OK], float)
DOM_REC = int(REC_OK[int(np.argmax(s_all))])
NB_TOT = int(sum(len(IDXS[r]) for r in REC_OK))
run.log(f"  레코드 {len(RS)} · 채점 가능 **{NRE}** · 총 박동 {NB_TOT} · 평균 유병률 {MEAN_PREV:.4f}")

EPS = 1e-6
def logit(p):
    p = np.clip(np.asarray(p, float), 1e-12, 1 - 1e-12)
    return np.log(p) - np.log1p(-p)

SLOPES = []
def make_cal(s, y):
    lr = LogisticRegression(max_iter=3000, C=1e6).fit(np.asarray(s).reshape(-1, 1),
                                                       np.asarray(y).astype(int))
    a, b = float(lr.coef_[0, 0]), float(lr.intercept_[0])
    SLOPES.append(a)
    return (lambda v: 1.0 / (1.0 + np.exp(-(a * np.asarray(v, float) + b))),
            lambda v: a * np.asarray(v, float) + b)

def em_prior(p, pi_tr, iters=100, tol=1e-9, clip=1e-2):
    pi = float(pi_tr)
    for _ in range(int(iters)):
        w = pi / pi_tr; v = (1.0 - pi) / (1.0 - pi_tr)
        num = w * p
        pp = num / (num + v * (1.0 - p))
        new = float(np.clip(pp.mean(), clip, 1.0 - clip))
        if abs(new - pi) < tol:
            pi = new; break
        pi = new
    return pi

def split_rest(held):
    rest = sorted([r for r in REC_OK if r != held], key=lambda r: (BURD[r], r))
    dv = [r for i, r in enumerate(rest) if i % DEV_EVERY == 0]
    return [r for r in rest if r not in set(dv)], dv

def loro(arm, y_override=None):
    """반환: (보정 로짓, 보정 확률, fold 별 DEV 자료) — 문턱·TPR/FPR 을 DEV 에서만 뽑는다."""
    out = np.full(len(K), np.nan); prob = np.full(len(K), np.nan); dev = {}
    for held in REC_OK:
        tr_r, dv_r = split_rest(held)
        tr = np.concatenate([IDXS[r] for r in tr_r]); dv = np.concatenate([IDXS[r] for r in dv_r])
        te = IDXS[held]
        use_b = (arm == "TE")
        if use_b:
            mu = float(np.mean([BURD[r] for r in tr_r]))
            Ftr = np.c_[RHY[tr], np.array([BURD[int(r)] for r in RID[tr]], float)]
            feat = lambda ii, b_: np.c_[RHY[ii], b_]
        else:
            Ftr = RHY[tr]; feat = lambda ii, b_: RHY[ii]
        fmu, fsd = Ftr.mean(0), Ftr.std(0) + 1e-9
        ytr = TT_[tr].astype(int) if y_override is None else y_override[held]
        lr = LogisticRegression(max_iter=3000, C=1.0).fit((Ftr - fmu) / fsd, ytr)
        sc = lambda ii, b_=None: lr.decision_function((feat(ii, b_) - fmu) / fsd)
        s_dv = (sc(dv, np.array([BURD[int(r)] for r in RID[dv]], float)) if use_b else sc(dv))
        cp, cl = make_cal(s_dv, TT_[dv])
        pi_tr = float(TT_[dv].mean())
        if use_b:
            bh = em_prior(cp(sc(te, np.full(len(te), mu))), pi_tr)
            s_te = sc(te, np.full(len(te), bh))
        else:
            s_te = sc(te)
        l = cl(s_te)
        if arm == "A_em":
            l = l + (logit(em_prior(cp(s_te), pi_tr)) - logit(pi_tr))
        out[te] = l; prob[te] = 1.0 / (1.0 + np.exp(-np.clip(l, -60, 60)))
        dev[held] = dict(logit=cl(s_dv), y=TT_[dv].astype(int), pi_tr=pi_tr)
    return out, prob, dev

# ── 배분 방식 둘
def alloc_budget(L, k_of):
    """★ **환자별 예산** — 각 레코드에서 상위 k개(문턱 없음 = 순수 순위)."""
    out = {}
    for r in REC_OK:
        pos = IDXS[r]; sc = L[pos]; yy = TT_[pos]
        k = int(min(max(1, k_of(r)), len(pos)))
        thr = np.partition(sc, -k)[-k]
        fl = sc >= thr
        tp = int((fl & yy).sum())
        out[r] = dict(sens=tp / max(1, int(yy.sum())), ppv=tp / max(1, int(fl.sum())),
                      tp=tp, fp=int((fl & ~yy).sum()), fn=int(yy.sum()) - tp,
                      flagged=int(fl.sum()), n=len(pos), prev=BURD[r])
    return out

def alloc_global(L, dev, q):
    """★★ **전역 단일 문턱** — 그 fold 의 DEV 에서만 잡는다(R22)."""
    out = {}
    for r in REC_OK:
        thr = float(np.quantile(dev[r]["logit"], 1.0 - q))
        pos = IDXS[r]; sc = L[pos]; yy = TT_[pos]
        fl = sc >= thr
        tp = int((fl & yy).sum())
        out[r] = dict(sens=tp / max(1, int(yy.sum())),
                      ppv=(tp / int(fl.sum())) if fl.sum() else np.nan,
                      tp=tp, fp=int((fl & ~yy).sum()), fn=int(yy.sum()) - tp,
                      flagged=int(fl.sum()), n=len(pos), rate=float(fl.mean()),
                      thr=thr, prev=BURD[r])
    return out

run.log("  배분 방식 정의 완료 — **환자별 예산**(상위 k개) vs **전역 단일 문턱**(DEV 분위수)")
CONFIG["cohort"] = dict(n_rec=len(RS), n_ok=NRE, n_beat=NB_TOT, mean_prev=MEAN_PREV,
                        dom_rec=DOM_REC, prev_min=min(BURD.values()),
                        prev_max=max(BURD.values()))
run.save_json("config", CONFIG)


In [ ]:
# CELL 3 — 【J-A】 실행 · J0 · ★★★ J2 층② 무관성(자)
run.log("\n" + "=" * 100)
run.log("【J-A】 실행 · J0(Platt 기울기) · ★★★ J2(예산 방식에서 층② 가 무의미한가)")
run.log("=" * 100)
T0 = time.time()
L, P, DEV = {}, {}, {}
for a in ARMS:
    L[a], P[a], DEV[a] = loro(a)
    run.log(f"  ({time.time()-T0:>5.0f}초) {a} 완료")

AP = {a: {r: float(average_precision_score(TT_[IDXS[r]].astype(int), L[a][IDXS[r]]))
          for r in REC_OK} for a in ARMS}
AUC = {a: {r: float(roc_auc_score(TT_[IDXS[r]].astype(int), L[a][IDXS[r]]))
           for r in REC_OK} for a in ARMS}
run.log(f"\n  {'팔':<8}{'매크로 PR-AUC':>15}{'매크로 AUROC':>15}")
for a in ARMS:
    run.log(f"  {a:<8}{np.mean(list(AP[a].values())):>15.4f}"
            f"{np.mean(list(AUC[a].values())):>15.4f}")
run.log(f"  (Q4-E 앵커 — 매크로 PR-AUC {REF['macro_ap']} · AUROC {REF['macro_auc']})")

# ── J0
sl = np.array(SLOPES, float); neg = int((sl <= 0).sum()); frac = neg / max(1, len(sl))
run.log(f"\n  J0 — Platt 기울기 {len(sl)}개 · 중앙 {np.median(sl):+.4f} · 음수 {neg}({frac:.1%})")
if np.median(sl) <= 0 or frac > MAX_NEG_SLOPE:
    raise AssetError(f"J0 실패 — 중앙 {np.median(sl):.4f} · 음수 {frac:.1%}(R29 ②)")
g_("J0", "✅ 지지", f"Platt 기울기 중앙 {np.median(sl):+.4f} · 음수 {neg}/{len(sl)} — "
                    "체계적 반전 없음")

# ── ★★★ J2 — 예산 방식에서 층② 는 아무것도 못 한다(구성)
run.log("\n  ★★★ J2 — **예산 방식**에서 `raw` 와 `A_em` 의 레코드별 민감도")
run.log("     `A_em` 은 raw 의 **레코드별 상수 시프트**이고, 예산은 레코드 **내 순위만** 쓴다")
run.log("     → 정확히 같아야 한다. 서면 **층② 를 처방에서 통째로 뺄 수 있다**")
kq = lambda q: (lambda r: int(round(q * len(IDXS[r]))))
B = {q: {a: alloc_budget(L[a], kq(q)) for a in ARMS} for q in FLAG_Q}
d2 = max(abs(B[q][ "raw"][r]["sens"] - B[q]["A_em"][r]["sens"])
         for q in FLAG_Q for r in REC_OK)
d2p = max(abs(B[q]["raw"][r]["ppv"] - B[q]["A_em"][r]["ppv"])
          for q in FLAG_Q for r in REC_OK)
run.log(f"     max|Δ민감도| **{d2:.2e}** · max|ΔPPV| **{d2p:.2e}** (허용 {TOL_IDENT:.0e})")
if max(d2, d2p) >= TOL_IDENT:
    raise AssetError(f"J2 실패({max(d2, d2p):.3e}) — 예산은 레코드 내 순위만 쓰고 A_em 은 "
                     "레코드별 상수 시프트이므로 정확히 같아야 한다. 구현 오류다(R29 ②)")
d2te = float(np.mean([abs(B[MAIN_Q]["raw"][r]["sens"] - B[MAIN_Q]["TE"][r]["sens"])
                      for r in REC_OK]))
g_("J2", "✅ 지지",
   f"예산 방식에서 `raw` ≡ `A_em` 이 **정확히** 성립한다({d2:.1e}) — ★★★ **사전확률 "
   f"보정은 이 배포 방식에서 아무 일도 하지 않는다.** `TE`(부담 특징) 도 평균 "
   f"{d2te:.4f} 차이뿐이다 → **처방에서 층② 를 통째로 뺄 수 있다**")
CONFIG["J0"] = dict(slope_med=float(np.median(sl)), n_neg=neg, frac_neg=float(frac))
CONFIG["J2"] = dict(max_abs_sens=float(d2), max_abs_ppv=float(d2p), te_mean_abs=d2te,
                    macro_ap={a: float(np.mean(list(AP[a].values()))) for a in ARMS},
                    macro_auc={a: float(np.mean(list(AUC[a].values()))) for a in ARMS})
run.save_json("config", CONFIG)


In [ ]:
# CELL 4 — 【J-B】 ★★★ J1 주 관문 — 같은 총 경보 예산에서 배분 방식 비교
run.log("\n" + "=" * 100)
run.log("【J-B】 ★★★ J1 — **같은 총 경보 수**에서 환자별 배분 vs 전역 문턱")
run.log("=" * 100)
run.log("  ★ 총 경보 수를 맞춘다 — 안 맞추면 **많이 울리는 쪽이 민감도로 이긴다**(R36 ②)")
a0 = MAIN_ARM
G = {q: alloc_global(L[a0], DEV[a0], q) for q in FLAG_Q}

def matched_budget(L_, gstat):
    """전역 문턱이 코호트 전체에서 쓴 **총 경보 수**를 레코드 크기에 비례해 균등 배분한다."""
    tot = int(sum(gstat[r]["flagged"] for r in REC_OK))
    frac = tot / max(1, NB_TOT)
    return alloc_budget(L_, lambda r: int(round(frac * len(IDXS[r])))), tot, frac

J1 = {}
for q in FLAG_Q:
    Bm, tot_g, frac_g = matched_budget(L[a0], G[q])
    tot_b = int(sum(Bm[r]["flagged"] for r in REC_OK))
    ks = [r for r in REC_OK if np.isfinite(G[q][r]["sens"])]
    m_, lo_, hi_, n_ = boot_pair([G[q][r]["sens"] for r in ks], [Bm[r]["sens"] for r in ks],
                                 SEED0 + 41 + int(q * 100), NB_BOOT)
    tp_g = int(sum(G[q][r]["tp"] for r in REC_OK)); tp_b = int(sum(Bm[r]["tp"] for r in REC_OK))
    fn_g = int(sum(G[q][r]["fn"] for r in REC_OK)); fn_b = int(sum(Bm[r]["fn"] for r in REC_OK))
    J1[q] = dict(mean=m_, lo=lo_, hi=hi_, n=int(n_), mde=float(mde(lo_, hi_)),
                 tot_global=tot_g, tot_budget=tot_b, matched_frac=float(frac_g),
                 tp_global=tp_g, tp_budget=tp_b, fn_global=fn_g, fn_budget=fn_b,
                 sens_global=float(np.mean([G[q][r]["sens"] for r in ks])),
                 sens_budget=float(np.mean([Bm[r]["sens"] for r in ks])),
                 sd_global=float(np.std([G[q][r]["sens"] for r in ks], ddof=1)),
                 sd_budget=float(np.std([Bm[r]["sens"] for r in ks], ddof=1)),
                 min_global=float(np.min([G[q][r]["sens"] for r in ks])),
                 min_budget=float(np.min([Bm[r]["sens"] for r in ks])),
                 ppv_global=float(np.nanmean([G[q][r]["ppv"] for r in ks])),
                 ppv_budget=float(np.nanmean([Bm[r]["ppv"] for r in ks])))
    globals()[f"BM_{int(q*100)}"] = Bm
    j = J1[q]
    run.log(f"\n  ── 전역 문턱 q={q:.0%} 가 쓴 총 경보 **{tot_g}** → 같은 수를 환자별로 "
            f"배분(기록당 {frac_g:.2%} · 총 {tot_b})")
    run.log(f"  {'방식':<14}{'민감도 평균':>12}{'SD':>8}{'최소':>8}{'PPV':>9}"
            f"{'총 TP':>9}{'총 FN':>9}")
    run.log(f"  {'전역 문턱':<14}{j['sens_global']:>12.4f}{j['sd_global']:>8.4f}"
            f"{j['min_global']:>8.4f}{j['ppv_global']:>9.4f}{tp_g:>9d}{fn_g:>9d}")
    run.log(f"  {'환자별 배분':<14}{j['sens_budget']:>12.4f}{j['sd_budget']:>8.4f}"
            f"{j['min_budget']:>8.4f}{j['ppv_budget']:>9.4f}{tp_b:>9d}{fn_b:>9d}")
    run.log(f"    Δ(배분−전역) **환자 평균 민감도 {m_:+.4f}** [{lo_:+.4f}, {hi_:+.4f}] · "
            f"놓친 S 총계 {fn_g:d} → {fn_b:d} ({fn_g - fn_b:+d})")
    # ★★ 두 수가 **반대로 갈 수 있다** — 숨기지 말고 명시한다(R38 ⑦).
    #    환자 평균 민감도는 레코드를 **동등 가중**(R11)하고, 총 FN 은 **양성 수로 가중**한다.
    #    전역 문턱은 S 가 많은 레코드에 경보를 몰아주므로 총 FN 에서 유리할 수 있지만,
    #    그 대가로 **작은 레코드에서 0 을 준다**(Q4-E 에서 민감도 0 인 레코드가 나왔다).
    if (m_ > 0) != (fn_g - fn_b > 0):
        run.log(f"    ⚠️ **두 수가 반대 방향이다** — 환자 평균 민감도는 레코드를 동등 "
                f"가중하고(R11) 총 FN 은 **양성 수로 가중**하기 때문이다. 전역 문턱은 S 가 "
                f"많은 레코드에 경보를 몰아 총 FN 에서 유리하지만, 그 대가로 **작은 "
                f"레코드에 0 을 준다**(최소 민감도 {j['min_global']:.4f} vs "
                f"{j['min_budget']:.4f}). **R11 이 주 지표로 요구한 건 환자 단위**다")

# ── ★★ 영점
run.log(f"\n  ★★ **대비의 영점** — 학습 라벨 치환 (reps={N_PERM})")
q = MAIN_Q; nul = {}
for s_ in range(N_PERM):
    rr = np.random.RandomState(SEED0 + 400 + s_)
    yov = {}
    for held in REC_OK:
        tr_r, _ = split_rest(held)
        tr = np.concatenate([IDXS[r] for r in tr_r])
        yov[held] = TT_[tr].astype(int)[rr.permutation(len(tr))]
    ln, _pn, dn = loro(a0, y_override=yov)
    gn = alloc_global(ln, dn, q)
    bn, _t, _f = matched_budget(ln, gn)
    for r in REC_OK:
        if np.isfinite(gn[r]["sens"]):
            nul.setdefault(r, []).append(bn[r]["sens"] - gn[r]["sens"])
    run.log(f"    ({time.time()-T0:>5.0f}초) 영점 rep {s_+1}/{N_PERM}")
NS = boot_mean([float(np.mean(v)) for v in nul.values()], SEED0 + 61, NB_BOOT)
run.log(f"    영점 **{NS[0]:+.4f}** [{NS[1]:+.4f}, {NS[2]:+.4f}] (레코드 {NS[3]})")
J1_THR = max(0.0, NS[2]) if np.isfinite(NS[2]) else float("nan")
run.log(f"    ▸ 문턱 = max(0, 영점 상단) **{J1_THR:+.4f}**")
j1v = decide(J1[q]["lo"], J1[q]["hi"], J1_THR, ">")
g_("J1", j1v,
   f"**같은 총 경보 수**({J1[q]['tot_global']}개)에서 환자별 배분이 민감도 "
   f"{J1[q]['mean']:+.4f} [{J1[q]['lo']:+.4f}, {J1[q]['hi']:+.4f}] · 놓친 S "
   f"**{J1[q]['fn_global'] - J1[q]['fn_budget']:+d}개** · 문턱 {J1_THR:+.4f} — "
   + ("**환자별 배분이 배포 방식이다**" if j1v.startswith("✅") else
      ("전역 문턱이 낫다 — 층②(척도 정렬)가 다시 필요하다" if j1v.startswith("❌")
       else "가르지 못했다(R33 ①)")))
CONFIG["J1"] = dict(per_q={str(k): v for k, v in J1.items()},
                    null=dict(mean=NS[0], lo=NS[1], hi=NS[2], n=int(NS[3])),
                    thr=float(J1_THR))
run.save_json("config", CONFIG)


In [ ]:
# CELL 5 — 【J-C】 ★★ J3 임상 예산 · ★★★ J4 부담 정량
run.log("\n" + "=" * 100)
run.log("【J-C】 J3(임상 예산 · 절대 개수) · ★★★ J4(부담 정량 — π̂ 오차)")
run.log("=" * 100)
a0 = MAIN_ARM
run.log(f"  ★★ 24시간 홀터는 박동이 10만 개 규모다 — **5% 면 5000개**로 판독 불가능하다.")
run.log(f"     기록당 **절대 {FLAG_K}개**로 다시 잰다(이 코호트는 기록당 "
        f"{NB_TOT // NRE} 박동 · 30분 기록이다)")
BK = {k: alloc_budget(L[a0], lambda r, k=k: k) for k in FLAG_K}
run.log(f"\n  {'예산':<14}{'민감도 평균':>12}{'SD':>8}{'최소':>8}{'PPV 평균':>10}"
        f"{'총 TP':>9}{'총 FN':>9}")
J3 = {}
for k in FLAG_K:
    sv = np.array([BK[k][r]["sens"] for r in REC_OK], float)
    pv = np.array([BK[k][r]["ppv"] for r in REC_OK], float)
    J3[k] = dict(sens=float(sv.mean()), sd=float(sv.std(ddof=1)), min=float(sv.min()),
                 ppv=float(np.nanmean(pv)),
                 tp=int(sum(BK[k][r]["tp"] for r in REC_OK)),
                 fn=int(sum(BK[k][r]["fn"] for r in REC_OK)))
    run.log(f"  기록당 {k:<7}{sv.mean():>12.4f}{sv.std(ddof=1):>8.4f}{sv.min():>8.4f}"
            f"{np.nanmean(pv):>10.4f}{J3[k]['tp']:>9d}{J3[k]['fn']:>9d}")
run.log(f"  ▸ 유병률 사분위별 민감도(기록당 {FLAG_K[-1]}개)")
qs = np.quantile([BURD[r] for r in REC_OK], [0, .25, .5, .75, 1.0])
J3BAND = []
for i in range(4):
    lo_, hi_ = qs[i], qs[i + 1]
    rs = [r for r in REC_OK if (lo_ <= BURD[r] <= hi_ if i == 3 else lo_ <= BURD[r] < hi_)]
    if not rs:
        continue
    sv = float(np.mean([BK[FLAG_K[-1]][r]["sens"] for r in rs]))
    pv = float(np.nanmean([BK[FLAG_K[-1]][r]["ppv"] for r in rs]))
    J3BAND.append(dict(lo=float(lo_), hi=float(hi_), n=len(rs), sens=sv, ppv=pv,
                       prev=float(np.mean([BURD[r] for r in rs]))))
    run.log(f"    [{lo_:.4f}, {hi_:.4f}]  n={len(rs):<3} 유병률 "
            f"{J3BAND[-1]['prev']:.4f} · 민감도 {sv:.4f} · PPV {pv:.4f}")
g_("J3", "(관문 아님)",
   f"기록당 {FLAG_K[0]}개 — 민감도 {J3[FLAG_K[0]]['sens']:.4f} · PPV {J3[FLAG_K[0]]['ppv']:.4f} | "
   f"{FLAG_K[1]}개 — 민감도 {J3[FLAG_K[1]]['sens']:.4f} · PPV {J3[FLAG_K[1]]['ppv']:.4f}")

# ── ★★★ J4 — 부담 정량. 이 퀘스트가 **한 번도 안 잰 것**이다
run.log("\n  ★★★ J4 — **부담 정량(π̂ 오차)**")
run.log("     유병률 57.6% 환자에게 「이 박동이 SVEB 인가」는 **이미 답이 난 질문**이다.")
run.log("     필요한 건 **「SVEB 부담 X%」** — 그런데 이 퀘스트는 π̂ 오차를 지표로 잰 적이 없다")
PIH = {e: {} for e in PI_EST}
for r in REC_OK:
    d = DEV[a0][r]; te = IDXS[r]
    p_te = P[a0][te]; pi_tr = d["pi_tr"]
    PIH["mean_p"][r] = float(np.mean(p_te))
    PIH["em"][r] = float(em_prior(p_te, pi_tr))
    thr = float(np.quantile(d["logit"], 1.0 - MAIN_Q))
    rate = float((L[a0][te] >= thr).mean())
    PIH["count"][r] = rate
    dy = d["y"].astype(bool); dl = d["logit"]
    tpr = float((dl[dy] >= thr).mean()) if dy.any() else np.nan
    fpr = float((dl[~dy] >= thr).mean()) if (~dy).any() else np.nan
    PIH["bbse"][r] = (float(np.clip((rate - fpr) / (tpr - fpr), 0.0, 1.0))
                      if np.isfinite(tpr) and np.isfinite(fpr) and abs(tpr - fpr) > 1e-6
                      else np.nan)
run.log(f"\n  {'추정기':<10}{'평균 |π̂−π*|':>14}{'중앙':>9}{'최대':>9}"
        f"{'상대오차 중앙':>14}{'ρ(π̂,π*)':>11}")
J4 = {}
for e in PI_EST:
    v = np.array([PIH[e][r] for r in REC_OK], float)
    t = np.array([BURD[r] for r in REC_OK], float)
    m = np.isfinite(v)
    ae = np.abs(v[m] - t[m]); rel = ae / t[m]
    J4[e] = dict(mae=float(ae.mean()), med=float(np.median(ae)), max=float(ae.max()),
                 rel_med=float(np.median(rel)),
                 rho=float(np.corrcoef(v[m], t[m])[0, 1]) if m.sum() > 2 else np.nan,
                 n=int(m.sum()))
    run.log(f"  {e:<10}{J4[e]['mae']:>14.4f}{J4[e]['med']:>9.4f}{J4[e]['max']:>9.4f}"
            f"{J4[e]['rel_med']:>13.1%}{J4[e]['rho']:>11.4f}")
BEST = min(PI_EST, key=lambda e: J4[e]["mae"])
run.log(f"\n  ★ 최선 추정기 **`{BEST}`** — 평균 오차 {J4[BEST]['mae']:.4f} · "
        f"상대오차 중앙 {J4[BEST]['rel_med']:.1%}")
run.log(f"  ★ 지배 레코드 {DOM_REC}(π* {BURD[DOM_REC]:.4f}) — " +
        " · ".join(f"{e} {PIH[e][DOM_REC]:.4f}" for e in PI_EST))
run.log(f"  ▸ 고유병률 4분위(π* ≥ {qs[3]:.4f})에서의 오차")
hi_r = [r for r in REC_OK if BURD[r] >= qs[3]]
for e in PI_EST:
    v = np.array([PIH[e][r] for r in hi_r], float); t = np.array([BURD[r] for r in hi_r], float)
    m = np.isfinite(v)
    run.log(f"    {e:<10} 평균 |π̂−π*| {np.abs(v[m]-t[m]).mean():.4f} · 상대 "
            f"{np.median(np.abs(v[m]-t[m])/t[m]):.1%}")
g_("J4", "(관문 아님)",
   f"최선 `{BEST}` 평균 오차 {J4[BEST]['mae']:.4f}(상대 {J4[BEST]['rel_med']:.1%}) · "
   f"ρ {J4[BEST]['rho']:.4f} — **부담 정량을 산출물 지표로 처음 쟀다**")
CONFIG["J3"] = dict(by_k=J3, band=J3BAND)
CONFIG["J4"] = dict(err=J4, best=BEST,
                    pi_hat={e: {str(r): PIH[e][r] for r in REC_OK} for e in PI_EST},
                    pi_true={str(r): BURD[r] for r in REC_OK})
run.save_json("config", CONFIG)


In [ ]:
# CELL 6 — 【J-D】 J5 최악 환자 · 필요표본 · ★ J6 검산표
run.log("\n" + "=" * 100)
run.log("【J-D】 J5(자기 예산으로도 실패하는 환자) · 필요표본 · J6 검산표")
run.log("=" * 100)
a0 = MAIN_ARM; q = MAIN_Q
Bm = globals()[f"BM_{int(q*100)}"]
run.log(f"  J5 — **환자별 배분**에서도 민감도가 낮은 레코드(층② 로는 못 고친다 = 층① 과제)")
worst = sorted(REC_OK, key=lambda r: Bm[r]["sens"])[:8]
run.log(f"  {'레코드':<8}{'유병률':>9}{'민감도':>9}{'PPV':>8}{'AUROC':>9}{'PR-AUC':>9}"
        f"{'S 총수':>8}{'FN':>7}")
for r in worst:
    run.log(f"  #{r:<7}{BURD[r]:>9.4f}{Bm[r]['sens']:>9.4f}{Bm[r]['ppv']:>8.4f}"
            f"{AUC[a0][r]:>9.4f}{AP[a0][r]:>9.4f}"
            f"{int(TT_[IDXS[r]].sum()):>8d}{Bm[r]['fn']:>7d}")
sv = np.array([Bm[r]["sens"] for r in REC_OK], float)
av = np.array([AUC[a0][r] for r in REC_OK], float)
pv = np.array([BURD[r] for r in REC_OK], float)
run.log(f"\n    민감도~AUROC 상관 {np.corrcoef(sv, av)[0,1]:+.4f} · "
        f"민감도~유병률 상관 {np.corrcoef(sv, pv)[0,1]:+.4f}")
run.log(f"    민감도 0.20 미만 **{int((sv < 0.20).sum())}/{NRE}** 레코드 — 이들의 "
        f"AUROC 중앙 {np.median(av[sv < 0.20]) if (sv < 0.20).any() else float('nan'):.4f}")
g_("J5", "(관문 아님)",
   f"환자별 배분에서도 민감도 0.20 미만이 {int((sv < 0.20).sum())}/{NRE} — "
   f"민감도~AUROC 상관 {np.corrcoef(sv, av)[0,1]:+.4f}. **층② 로 못 고치는 자리**이므로 "
   "층①(표현)의 과제다")

run.log(f"\n  필요표본 (**관문 문턱 기준** · 레코드 · 현재 {NRE})")
eff = J1[q]["mean"] - J1_THR
n5 = need_super(NRE, J1[q]["mde"], eff); n8 = need_super(NRE, J1[q]["mde"], eff, True)
bad = (not np.isfinite(eff)) or abs(eff) < J1[q]["mde"]
run.log(f"  J1 민감도  효과-문턱 {eff:+.4f} · 반폭 {J1[q]['mde']:.4f} · n(50%) {n5:.0f} · "
        f"n(80%) {n8:.0f}  "
        + ("★ **해석 불가**(R41 ②)" if bad else
           ("이미 충분하다" if n8 <= NRE else "표본이 더 필요하다")))

run.log("\n  ★ J6 — **결론 검산표**")
CHECK = [
    dict(claim=f"J2 — 예산 방식에서 `raw` ≡ `A_em` 이 정확히 성립한다"
               f"({CONFIG['J2']['max_abs_sens']:.1e}) → {VERD['J2']}",
         num=f"`A_em` 은 raw 의 **레코드별 상수 시프트**이고 예산은 레코드 **내 순위만** 쓴다. "
             f"`TE` 도 평균 {CONFIG['J2']['te_mean_abs']:.4f} 차이뿐",
         assume="**없음** — 구성이고 런타임 검사한다",
         iffalse="★★★ 이것이 **처방에서 층② 를 빼도 되는 근거**다 — 사전확률 보정도, "
                 "부담 특징도, π̂ 추정도 이 배포 방식에선 필요 없다"),
    dict(claim=f"J1 — 같은 총 경보 수({J1[q]['tot_global']}개)에서 환자별 배분이 민감도 "
               f"{J1[q]['mean']:+.4f} [{J1[q]['lo']:+.4f}, {J1[q]['hi']:+.4f}] → {VERD['J1']}",
         num=f"놓친 S {J1[q]['fn_global']} → {J1[q]['fn_budget']}"
             f"({J1[q]['fn_global'] - J1[q]['fn_budget']:+d}) · 영점 "
             f"{CONFIG['J1']['null']['mean']:+.4f} · 문턱 {J1_THR:+.4f}",
         assume="**총 경보 수를 맞췄다** — 안 맞추면 많이 울리는 쪽이 이긴다(R36 ②)",
         iffalse="★ Q4-E 스모크에서 `A_em` 이 경보율 0.2505 로 민감도가 높아 보인 게 그 사례다"),
    dict(claim=f"J3 임상 예산 — 기록당 {FLAG_K[0]}개에서 민감도 "
               f"{CONFIG['J3']['by_k'][FLAG_K[0]]['sens']:.4f} · PPV "
               f"{CONFIG['J3']['by_k'][FLAG_K[0]]['ppv']:.4f}",
         num=f"이 코호트는 기록당 {NB_TOT // NRE} 박동(30분)이다 — 24시간이면 박동이 "
             f"약 {24 * 2 * (NB_TOT // NRE) // 1000}k 라 5% 는 판독 불가능하다",
         assume="검토 부담이 **기록당 절대 개수**로 정해진다는 것(홀터 과판독 워크플로)",
         iffalse="★ 분율 예산은 긴 기록에서 경보가 선형으로 늘어 실무와 안 맞는다"),
    dict(claim=f"J4 부담 정량 — 최선 `{CONFIG['J4']['best']}` 평균 |π̂−π*| "
               f"{CONFIG['J4']['err'][CONFIG['J4']['best']]['mae']:.4f}",
         num=f"상대오차 중앙 {CONFIG['J4']['err'][CONFIG['J4']['best']]['rel_med']:.1%} · "
             f"ρ {CONFIG['J4']['err'][CONFIG['J4']['best']]['rho']:.4f} · 네 추정기 비교",
         assume="**없음** — π* 는 라벨에서 직접 온다",
         iffalse="★★★ 고유병률 환자의 임상 과제는 **탐지가 아니라 정량**이다. 이 퀘스트가 "
                 "한 번도 안 잰 지표이고, 여기서 처음 쟀다"),
    dict(claim=f"고유병률에서 lift 가 낮은 건 **모델 열화가 아니다**",
         num=f"Q4-E 사분위별 AUROC {REF['qband_auc']} — 범위 "
             f"{max(REF['qband_auc']) - min(REF['qband_auc']):.4f} 로 평평하다. "
             f"레코드 {DOM_REC} 도 AUROC {REF['dom48']['auc']}",
         assume="**없음** — 같은 점수에서 두 지표를 냈다",
         iffalse="★★ PR-AUC 의 무작위 기저선이 유병률이라 lift 가 떨어지는 것이다. "
                 "다만 **그 환자에게 탐지의 임상적 가치가 낮은 것은 사실**이므로 과제를 "
                 "정량으로 바꾸는 게 옳은 대응이다"),
]
for i, ck in enumerate(CHECK, 1):
    run.log(f"\n  [{i}] **{ck['claim']}**")
    run.log(f"      근거   {ck['num']}")
    run.log(f"      가정   {ck['assume']}")
    run.log(f"      틀리면 {ck['iffalse']}")
CONFIG["J5"] = dict(worst=[dict(rec=r, prev=BURD[r], sens=Bm[r]["sens"], auc=AUC[a0][r],
                                ap=AP[a0][r], fn=Bm[r]["fn"]) for r in worst],
                    n_below_020=int((sv < 0.20).sum()),
                    rho_sens_auc=float(np.corrcoef(sv, av)[0, 1]),
                    rho_sens_prev=float(np.corrcoef(sv, pv)[0, 1]))
CONFIG["need"] = dict(j1=dict(effect=float(eff), half=float(J1[q]["mde"]),
                              sup50=float(n5), sup80=float(n8), uninterpretable=bool(bad)))
CONFIG["J6"] = CHECK
run.save_json("config", CONFIG)


In [ ]:
# CELL 7 — 【J-E】 그림 · 요약 · 마무리
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display
a0 = MAIN_ARM; q = MAIN_Q
Bm = globals()[f"BM_{int(q*100)}"]
fig, ax = plt.subplots(1, 3, figsize=(16.5, 4.6))

pv = np.array([BURD[r] for r in REC_OK])
ax[0].scatter(pv, [G[q][r]["sens"] for r in REC_OK], s=30, color="tab:gray",
              label="global threshold")
ax[0].scatter(pv, [Bm[r]["sens"] for r in REC_OK], s=30, color="tab:blue", marker="^",
              label="per-patient (matched total)")
ax[0].annotate(f"#{DOM_REC}", (BURD[DOM_REC], G[q][DOM_REC]["sens"]), fontsize=8,
               xytext=(4, 4), textcoords="offset points")
ax[0].set_xlabel("record prevalence"); ax[0].set_ylabel(f"sensitivity")
ax[0].set_title("J1 : same total alarms, different allocation", fontsize=9)
ax[0].legend(fontsize=7); ax[0].grid(alpha=.3)

xs = np.arange(len(PI_EST))
ax[1].bar(xs, [CONFIG["J4"]["err"][e]["mae"] for e in PI_EST], color="tab:orange")
ax[1].set_xticks(xs); ax[1].set_xticklabels(PI_EST, fontsize=8)
ax[1].set_ylabel("mean |pi_hat - pi_true|")
ax[1].set_title("J4 : burden quantification error", fontsize=9)
ax[1].grid(alpha=.3, axis="y")

for e, col in (("em", "tab:red"), (CONFIG["J4"]["best"], "tab:green")):
    ax[2].scatter([BURD[r] for r in REC_OK],
                  [CONFIG["J4"]["pi_hat"][e][str(r)] for r in REC_OK], s=26, label=e,
                  color=col, alpha=.8)
ax[2].plot([0, pv.max()], [0, pv.max()], "k--", lw=1.0)
ax[2].set_xlabel("true burden"); ax[2].set_ylabel("estimated burden")
ax[2].set_title("J4 : is the burden report usable", fontsize=9)
ax[2].legend(fontsize=7); ax[2].grid(alpha=.3)
fig.tight_layout()
PNG = run.save_fig("q4f_deployment_mode", fig)
plt.close(fig); display(Image(PNG))

run.log("\n" + "=" * 100)
run.log("요약")
run.log("=" * 100)
ok_ = lambda k: VERD.get(k, "").startswith("✅")
for g in READ_ORDER[:6]:
    run.log(f"  {g:<5}{VERD.get(g, '(관문 아님)')}")
run.log("")
j = J1[q]
run.log(f"  ★★★ **배포 방식** — 같은 총 경보 {j['tot_global']}개에서")
run.log(f"     전역 문턱   민감도 {j['sens_global']:.4f} (SD {j['sd_global']:.4f} · 최소 "
        f"{j['min_global']:.4f}) · PPV {j['ppv_global']:.4f} · 놓친 S {j['fn_global']}")
run.log(f"     환자별 배분 민감도 {j['sens_budget']:.4f} (SD {j['sd_budget']:.4f} · 최소 "
        f"{j['min_budget']:.4f}) · PPV {j['ppv_budget']:.4f} · 놓친 S {j['fn_budget']}")
run.log(f"     Δ **{j['mean']:+.4f}** [{j['lo']:+.4f}, {j['hi']:+.4f}] → {VERD['J1']}")
if (j["mean"] > 0) != (j["fn_global"] - j["fn_budget"] > 0):
    run.log(f"     ⚠️ 환자 평균 민감도와 총 FN 이 **반대 방향**이다 — 전자는 레코드 동등 "
            f"가중(R11), 후자는 양성 수 가중이다. 전역 문턱은 큰 레코드에 경보를 몰아 "
            f"총 FN 에서 유리하지만 **작은 레코드를 버린다**(최소 민감도 "
            f"{j['min_global']:.4f} vs {j['min_budget']:.4f})")
if ok_("J1") and ok_("J2"):
    run.log("")
    run.log(f"  ★★★ **처방 확정 — `{a0}` + Platt + 환자별 예산.**")
    run.log(f"     층② (사전확률 보정·부담 특징·π̂ 추정)를 **전부 뺀다** — 예산 방식에서 "
            f"`raw` ≡ `A_em` 이 정확히 성립한다({CONFIG['J2']['max_abs_sens']:.1e})")
run.log("")
run.log(f"  ★★ **임상 예산** — 기록당 {FLAG_K[0]}개 민감도 "
        f"{CONFIG['J3']['by_k'][FLAG_K[0]]['sens']:.4f}·PPV "
        f"{CONFIG['J3']['by_k'][FLAG_K[0]]['ppv']:.4f} | {FLAG_K[1]}개 "
        f"{CONFIG['J3']['by_k'][FLAG_K[1]]['sens']:.4f}·"
        f"{CONFIG['J3']['by_k'][FLAG_K[1]]['ppv']:.4f}")
run.log(f"  ★★★ **부담 정량** — 최선 `{CONFIG['J4']['best']}` 평균 |π̂−π*| "
        f"{CONFIG['J4']['err'][CONFIG['J4']['best']]['mae']:.4f} · 상대 "
        f"{CONFIG['J4']['err'][CONFIG['J4']['best']]['rel_med']:.1%} · "
        f"ρ {CONFIG['J4']['err'][CONFIG['J4']['best']]['rho']:.4f}")
run.log(f"     ▸ 고유병률 환자의 과제는 **탐지가 아니라 정량**이다 — 이 지표가 그 트랙의 자다")
run.log(f"  ★ **층① 과제** — 자기 예산으로도 민감도 0.20 미만이 "
        f"{CONFIG['J5']['n_below_020']}/{NRE} · 민감도~AUROC 상관 "
        f"{CONFIG['J5']['rho_sens_auc']:+.4f}")

run.finish({
    "exp_id": "quest46_q4f_deployment_mode",
    "metric": "budget_minus_global_sensitivity",
    "value": float(j["mean"]),
    "passed": bool(ok_("J0") and ok_("J1") and ok_("J2")),
    "summary": ("배포 방식을 판정했다. 같은 총 경보 수에서 **환자별 배분 vs 전역 단일 문턱**을 "
                "비교하고(J1), 예산 방식에서 층② 가 **구성으로 무의미**함을 확인했다"
                "(J2 — `raw` ≡ `A_em` 정확히). 그리고 24시간 홀터에서 판독 가능한 "
                "**절대 예산**(기록당 100·300개)으로 다시 쟀다(J3). ★★★ 고유병률 환자의 "
                "임상 과제는 탐지가 아니라 **정량**이므로, 이 퀘스트가 한 번도 안 잰 "
                "**π̂ 오차를 산출물 지표로 처음 측정**했다(J4). Q4-E 가 보인 대로 사분위별 "
                "AUROC 는 평평하므로(0.9340~0.9513) 고유병률의 낮은 lift 는 모델 열화가 "
                "아니라 PR-AUC 기저선 상승이다."),
    "verdicts": VERD, "notes": NOTE, "rule_check": RULE_CHECK,
    "cohort": CONFIG.get("cohort", {}), "J0": CONFIG.get("J0", {}),
    "J1": CONFIG.get("J1", {}), "J2": CONFIG.get("J2", {}), "J3": CONFIG.get("J3", {}),
    "J4": CONFIG.get("J4", {}), "J5": CONFIG.get("J5", {}),
    "need": CONFIG.get("need", {}), "J6": CONFIG.get("J6", []), "fig": PNG})
run.log(f"\n저장 완료 — {run.dir}")
run.log("다음: `python pipelines/ingest_run.py --results result.json "
        "--notebook notebooks/quest46_q4f_deployment_mode.ipynb`")
